# 论文 9：GPipe——使用流水线并行高效训练巨型神经网络

**论文**：Huang et al.（2019），GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism

**核心观点**：训练超大规模神经网络时，需要将模型拆分到多个设备。GPipe 将**流水线并行**、**微批处理**和**重计算**结合起来，从而高效训练单个加速器无法容纳的模型。

## 核心概念

### 1. 流水线并行
- 将模型拆分为跨 K 个设备的 **K 个分区**
- 每个设备拥有连续的层
- 数据依次流过流水线：设备 1 → 设备 2 → ... → 设备 K

### 2. 微批处理
- 将大小为 N 的小批次拆成 M 个大小为 N/M 的微批次
- 让多个微批次依次通过流水线
- **缩短流水线气泡时间**，也就是设备空闲时间

### 3. F-then-B 调度
```
Forward all M micro-batches, then backward all M micro-batches
Device 1: F1 F2 F3 F4 ........... B4 B3 B2 B1
Device 2: .. F1 F2 F3 F4 ....... B4 B3 B2 B1
Device 3: .... F1 F2 F3 F4 ..... B4 B3 B2 B1
Device 4: ...... F1 F2 F3 F4 ... B4 B3 B2 B1
```

### 4. 重计算（梯度检查点）
- 不保存全部激活值，因为这会占用大量内存
- 只在分区边界保存检查点
- 在反向传播时重新计算中间激活值
- **用额外计算换取更低的内存占用**

### 5. 流水线气泡时间
- 设备空闲时间比例：**(K-1) / (K-1 + M)**
- 微批次数量 M 越多 → 气泡时间越少
- 设备数量 K 越多 → 气泡时间越多

---

## 实现概览

我们将实施：
1. 跨“模拟”设备的模型分区
2. 微批次拆分与调度
3. 流水线中的前向传播和反向传播
4. 梯度累积
5. 使用重计算提高内存效率
6. 与数据并行进行比较
7. 分析流水线气泡时间

让我们来构建它吧！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Callable
from dataclasses import dataclass
import time
from collections import defaultdict

np.random.seed(42)

print("Libraries imported successfully!")
print("NumPy version:", np.__version__)

# 第 1 节：模型分区与流水线结构

GPipe 的第一步是将大型模型划分成 K 个分区，并将每个分区分配给不同设备。

## 分区策略

对于具有 L 层的模型：
- **均匀分区**：每个分区大约包含 L/K 层
- **平衡分区**：根据计算时间或内存占用划分

这里实现一个简单的多层网络，并对它进行均匀分区。

In [ ]:
@dataclass
class Layer:
    '单个神经网络层。'
    W: np.ndarray  # 权重矩阵
    b: np.ndarray  # 偏置向量
    activation: str = 'relu'  # “relu”、“tanh”或“线性”
    
    def forward(self, x: np.ndarray, store_activation: bool = True) -> Tuple[np.ndarray, np.ndarray]:
        '前向传播：z = W @ x + b，a = 激活(z)'
        z = x @ self.W + self.b  # 线性变换
        
        # 应用激活函数
        if self.activation == 'relu':
            a = np.maximum(0, z)
        elif self.activation == 'tanh':
            a = np.tanh(z)
        elif self.activation == 'linear':
            a = z
        else:
            raise ValueError(f"Unknown activation: {self.activation}")
        
        return a, z if store_activation else None
    
    def backward(self, da: np.ndarray, z: np.ndarray, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        '向后 pass：计算梯度。'
        # 激活梯度
        if self.activation == 'relu':
            dz = da * (z > 0)
        elif self.activation == 'tanh':
            dz = da * (1 - np.tanh(z)**2)
        elif self.activation == 'linear':
            dz = da
        else:
            raise ValueError(f"Unknown activation: {self.activation}")
        
        # 参数梯度
        dW = x.T @ dz
        db = np.sum(dz, axis=0)
        
        # 输入梯度（对于前一层）
        dx = dz @ self.W.T
        
        return dx, dW, db


@dataclass
class Partition:
    '模型的分区（分配给一个设备的层的子集）。'
    device_id: int
    layers: List[Layer]
    
    def forward(self, x: np.ndarray, store_activations: bool = True) -> Tuple[np.ndarray, List[Tuple]]:
        '前向传播该分区中的所有层。'
        activations = []  # 如果需要，为每层存储 (x, z)
        
        current = x
        for layer in self.layers:
            if store_activations:
                activations.append(current)  # 将输入存储到该层
            
            current, z = layer.forward(current, store_activation=store_activations)
            
            if store_activations:
                activations.append(z)  # 商店预激活
        
        return current, activations
    
    def backward(self, dout: np.ndarray, activations: List) -> Tuple[np.ndarray, List[Tuple]]:
        '向后遍历该分区中的所有层。'
        gradients = []  # 为每层存储（dW，db）
        
        da = dout
        # 反向遍历各层
        for i in range(len(self.layers) - 1, -1, -1):
            layer = self.layers[i]
            
            # 获取存储的激活
            x = activations[2*i]      # 该层的输入
            z = activations[2*i + 1]  # 预激活
            
            # 计算梯度
            da, dW, db = layer.backward(da, z, x)
            gradients.insert(0, (dW, db))
        
        return da, gradients  # da 是梯度 w.r.t。分区输入


def create_model(layer_dims: List[int], activations: List[str]) -> List[Layer]:
    """创建多层神经网络。
    
    参数：
        layer_dims：[input_dim, hidden1, hidden2, ..., output_dim]
        activations：每层激活"""
    layers = []
    for i in range(len(layer_dims) - 1):
        W = np.random.randn(layer_dims[i], layer_dims[i+1]) * np.sqrt(2.0 / layer_dims[i])
        b = np.zeros(layer_dims[i+1])
        layers.append(Layer(W, b, activations[i]))
    return layers


def partition_model(layers: List[Layer], num_partitions: int) -> List[Partition]:
    '跨设备统一分区层。'
    num_layers = len(layers)
    layers_per_partition = num_layers // num_partitions
    
    partitions = []
    for k in range(num_partitions):
        start = k * layers_per_partition
        if k == num_partitions - 1:
            # 最后一个分区获取所有剩余层
            end = num_layers
        else:
            end = (k + 1) * layers_per_partition
        
        partition_layers = layers[start:end]
        partitions.append(Partition(device_id=k, layers=partition_layers))
    
    return partitions


# 示例：创建并划分 12 层网络
layer_dims = [128] + [256] * 10 + [10]  # 输入 = 128，10 个隐藏层，共 256 个，输出 = 10
activations = ['relu'] * 10 + ['linear']  # ReLU 用于隐藏，线性用于输出

model_layers = create_model(layer_dims, activations)
print(f"Created model with {len(model_layers)} layers")

# 跨 4 个“设备”进行分区
K = 4
partitions = partition_model(model_layers, K)

print(f"\nPartitioned model into {K} partitions:")
for i, partition in enumerate(partitions):
    print(f"  Device {i}: {len(partition.layers)} layers")

print("\n✓ Model partitioning complete!")

# 第 2 节：微批处理策略

GPipe 将每个小批次拆分成 M 个**微批次**，以提高流水线利用率。

## 为什么要进行微批处理？

不使用微批处理时：
```
Device 1: [Forward] .................... [Backward]
Device 2:          [Forward] .......... [Backward]
Device 3:                   [Forward] [Backward]
          ^^^^^^^^                     ^^^^^^^^^^
          Bubble                       Bubble
```

使用 M 个微批次时：
```
Device 1: F1 F2 F3 F4 ........... B4 B3 B2 B1
Device 2:    F1 F2 F3 F4 ....... B4 B3 B2 B1
Device 3:       F1 F2 F3 F4 .... B4 B3 B2 B1
          ^^                              ^^
          Smaller bubble
```

**气泡占比**：(K-1) / (K-1 + M)
- 微批次越多，气泡时间越少
- 但微批次过多也会增加管理开销

In [ ]:
def split_into_microbatches(X: np.ndarray, y: np.ndarray, num_microbatches: int) -> List[Tuple[np.ndarray, np.ndarray]]:
    """将小批量拆分为微批次。
    
    参数：
        X：输入数据（batch_size，功能）
        y：标签（batch_size，...）
        num_microbatches：M（微批次数）
    
    返回：
        (X_micro、y_micro) 元组列表"""
    batch_size = X.shape[0]
    microbatch_size = batch_size // num_microbatches
    
    if batch_size % num_microbatches != 0:
        raise ValueError(f"Batch size {batch_size} must be divisible by num_microbatches {num_microbatches}")
    
    microbatches = []
    for m in range(num_microbatches):
        start = m * microbatch_size
        end = (m + 1) * microbatch_size
        microbatches.append((X[start:end], y[start:end]))
    
    return microbatches


def compute_bubble_fraction(K: int, M: int) -> float:
    """GPipe 的理论气泡占比。
    
    Formula: (K - 1) / (K - 1 + M)
    
    参数：
        K：设备/分区数量
        M：微批次数量"""
    return (K - 1) / (K - 1 + M)


# 示例：分析气泡占比
K_values = [2, 4, 8, 16]
M_values = [1, 2, 4, 8, 16, 32, 64]

print("Bubble Fraction Analysis:")
print("\nM (micro-batches) →")
print("K ↓\t" + "\t".join(f"{M:d}" for M in M_values))
print("-" * 80)

for K in K_values:
    row = f"{K}\t"
    for M in M_values:
        bubble = compute_bubble_fraction(K, M)
        row += f"{bubble:.3f}\t"
    print(row)

print("\nKey observations:")
print("  - More devices (K) → more bubble time (devices wait for pipeline)")
print("  - More micro-batches (M) → less bubble time (pipeline stays full)")
print("  - With K=4, M=8: bubble fraction = 27.3% (device idle 27% of time)")
print("  - With K=4, M=32: bubble fraction = 8.6% (much better!)")

# 微批处理示例
batch_size = 32
M = 8
X_batch = np.random.randn(batch_size, 128)
y_batch = np.random.randint(0, 10, batch_size)

microbatches = split_into_microbatches(X_batch, y_batch, M)
print(f"\n\nSplit batch of {batch_size} into {M} micro-batches:")
for i, (X_m, y_m) in enumerate(microbatches):
    print(f"  Micro-batch {i}: X shape {X_m.shape}, y shape {y_m.shape}")

print("\n✓ Micro-batching complete!")

# 第 3 节：流水线前向传播（F-then-B 调度）

GPipe 使用 **F-then-B 调度**：
1. 先让全部 M 个微批次完成前向传播
2. 再按相反顺序让全部 M 个微批次完成反向传播

## 时间线示例（K=3 个设备，M=4 个微批次）：

```
Time →  0   1   2   3   4   5   6   7   8   9   10  11  12
Dev 0:  F0  F1  F2  F3  ... ... ... B3  B2  B1  B0
Dev 1:  ... F0  F1  F2  F3  ... ... ... B3  B2  B1  B0
Dev 2:  ... ... F0  F1  F2  F3  ... ... ... B3  B2  B1  B0
```

图例：
- **F0** = 微批次 0 的前向传播
- **B3** = 微批次 3 的反向传播
- **...** = 流水线气泡（设备空闲）

In [ ]:
@dataclass
class PipelineEvent:
    '记录设备执行操作的时间。'
    time_step: int
    device_id: int
    operation: str  # “前进”或“后退”
    microbatch_id: int


class GPipePipeline:
    '采用 F-then-B 调度的 GPipe 流水线。'
    
    def __init__(self, partitions: List[Partition]):
        self.partitions = partitions
        self.K = len(partitions)  # 设备数量
        
        # 用于记录执行时间线
        self.events = []  # PipelineEvent 列表
    
    def forward_pipeline(self, microbatches: List[Tuple[np.ndarray, np.ndarray]], 
                        store_activations: bool = True) -> Tuple[List[np.ndarray], List[List]]:
        """前向传播：通过流水线处理所有微批次。
        
        返回：
            outputs：每个微批次的最终输出列表
            all_activations：激活列表列表（每个微批次一个）"""
        M = len(microbatches)
        
        # 输出和激活的存储
        outputs = [None] * M
        all_activations = [[None] * self.K for _ in range(M)]  # [微批次][分区]
        
        # F-then-B 调度：让所有微批次完成前向传播
        time_step = 0
        
        for m in range(M):
            X_micro, y_micro = microbatches[m]
            current = X_micro
            
            # 依次通过每个分区进行前向传播
            for k, partition in enumerate(self.partitions):
                self.events.append(PipelineEvent(time_step, k, 'forward', m))
                
                current, activations = partition.forward(current, store_activations)
                all_activations[m][k] = activations
                
                time_step += 1
            
            outputs[m] = current
        
        return outputs, all_activations
    
    def backward_pipeline(self, outputs: List[np.ndarray], 
                         labels: List[np.ndarray],
                         all_activations: List[List]) -> List[List[List[Tuple]]]:
        """向后 pass：反向处理所有微批次。
        
        返回：
            all_gradients：[微批次][分区][每层的（dW，db）]"""
        M = len(outputs)
        
        # 梯度存储
        all_gradients = [[None] * self.K for _ in range(M)]
        
        # 查找当前时间步（前向传播后）
        time_step = max(e.time_step for e in self.events) + 1
        
        # 以相反的顺序向后推所有微批次
        for m in range(M - 1, -1, -1):
            # 计算损失梯度（简单的MSE用于演示）
            dout = 2 * (outputs[m] - labels[m]) / labels[m].shape[0]
            
            # 反向向后遍历每个分区
            for k in range(self.K - 1, -1, -1):
                partition = self.partitions[k]
                activations = all_activations[m][k]
                
                self.events.append(PipelineEvent(time_step, k, 'backward', m))
                
                dout, gradients = partition.backward(dout, activations)
                all_gradients[m][k] = gradients
                
                time_step += 1
        
        return all_gradients
    
    def get_timeline_matrix(self) -> np.ndarray:
        """将事件转换为 K×T 矩阵以进行可视化。
        
        矩阵values：
            0 = 气泡（空闲）
            m+1 = 前向微批次 m
            -(m+1) = 向后微批次 m"""
        max_time = max(e.time_step for e in self.events) + 1
        timeline = np.zeros((self.K, max_time))
        
        for event in self.events:
            value = event.microbatch_id + 1
            if event.operation == 'backward':
                value = -value
            timeline[event.device_id, event.time_step] = value
        
        return timeline


# 测试前向传播
print("Testing GPipe forward pass...\n")

# 创建流水线
pipeline = GPipePipeline(partitions)

# 创建微批次
M = 4
batch_size = 16
X_batch = np.random.randn(batch_size, 128)
y_batch_onehot = np.eye(10)[np.random.randint(0, 10, batch_size)]

microbatches = split_into_microbatches(X_batch, y_batch_onehot, M)

# 前向传播
outputs, all_activations = pipeline.forward_pipeline(microbatches)

print(f"Processed {M} micro-batches through {pipeline.K} devices")
print(f"Output shapes: {[out.shape for out in outputs]}")
print(f"Total forward events: {len([e for e in pipeline.events if e.operation == 'forward'])}")

# 向后传球
labels = [mb[1] for mb in microbatches]
all_gradients = pipeline.backward_pipeline(outputs, labels, all_activations)

print(f"Total backward events: {len([e for e in pipeline.events if e.operation == 'backward'])}")
print(f"\nTotal time steps: {max(e.time_step for e in pipeline.events) + 1}")

print("\n✓ Pipeline forward and backward passes complete!")

# 第 4 节：跨微批次的梯度累积

处理完所有 M 个微批次后，我们需要：
1. **累积所有微批次的梯度**
2. 对它们取**平均**，因为这些梯度属于同一个小批次
3. **应用**累积的梯度来更新参数

这与一次处理整个小批次等价，但可以获得更高的流水线利用率。

In [ ]:
def accumulate_gradients(all_gradients: List[List[List[Tuple]]]) -> List[List[Tuple]]:
    """累积并平均所有微批次的梯度。
    
    参数：
        all_gradients：[微批次][分区][（dW，db）每层]
    
    返回：
        accumulated：[分区][（dW，db）每层] - 微批次的平均值"""
    M = len(all_gradients)  # 微批次数量
    K = len(all_gradients[0])  # 分区数量
    
    # 初始化累积梯度（从第一个微批次复制结构）
    accumulated = []
    for k in range(K):
        partition_grads = []
        for layer_idx in range(len(all_gradients[0][k])):
            # 跨微批次的梯度求和
            dW_sum = sum(all_gradients[m][k][layer_idx][0] for m in range(M))
            db_sum = sum(all_gradients[m][k][layer_idx][1] for m in range(M))
            
            # 平均值（因为微批次是同一小批次的一部分）
            dW_avg = dW_sum / M
            db_avg = db_sum / M
            
            partition_grads.append((dW_avg, db_avg))
        
        accumulated.append(partition_grads)
    
    return accumulated


def apply_gradients(partitions: List[Partition], gradients: List[List[Tuple]], learning_rate: float):
    """应用累积梯度来更新参数。
    
    参数：
        partitions：模型分区列表
        gradients: [分区][(dW, db) 每层]
        learning_rate：SGD 的学习率"""
    for k, partition in enumerate(partitions):
        partition_grads = gradients[k]
        
        for layer_idx, layer in enumerate(partition.layers):
            dW, db = partition_grads[layer_idx]
            
            # SGD 更新
            layer.W -= learning_rate * dW
            layer.b -= learning_rate * db


# 测试梯度累积
print("Testing gradient accumulation...\n")

# 我们已经有来自上一个单元的 all_gradients
accumulated_grads = accumulate_gradients(all_gradients)

print(f"Accumulated gradients for {len(accumulated_grads)} partitions:")
for k, partition_grads in enumerate(accumulated_grads):
    print(f"  Partition {k}: {len(partition_grads)} layers")
    for i, (dW, db) in enumerate(partition_grads[:2]):  # 显示前 2 层
        print(f"    Layer {i}: dW shape {dW.shape}, db shape {db.shape}")
        print(f"             dW norm: {np.linalg.norm(dW):.6f}, db norm: {np.linalg.norm(db):.6f}")

# 应用渐变
learning_rate = 0.01
old_W = partitions[0].layers[0].W.copy()

apply_gradients(partitions, accumulated_grads, learning_rate)

new_W = partitions[0].layers[0].W
weight_change = np.linalg.norm(new_W - old_W)

print(f"\nApplied gradients with learning rate {learning_rate}")
print(f"Weight change (first layer): {weight_change:.6f}")

print("\n✓ Gradient accumulation and application complete!")

# 第 5 节：重计算（梯度检查点）

**问题**：跨 K 个分区存储所有 M 个微批次的激活需要 O(M × K × layer_memory) 内存。

**解决方案**：使用**重计算**（梯度检查点）
- 只在**分区边界**保存激活值检查点
- 在反向传播期间**重新计算**中间激活值
- 代价：增加约 33% 的计算量，换取约 K 倍的内存节省

## 内存比较

**不使用重计算时**：
- 存储所有分区中所有层的激活
- 内存：O(M × L)，其中 L = 总层数

**使用重计算时**：
- 仅在分区边界存储激活
- 内存：O(M × K)，其中 K = 分区数 (K << L)
- 根据需要重新计算中间激活

In [ ]:
class GPipePipelineWithRemat:
    'GPipe 具有重计算（梯度检查点）。'
    
    def __init__(self, partitions: List[Partition]):
        self.partitions = partitions
        self.K = len(partitions)
        self.events = []
    
    def forward_pipeline_remat(self, microbatches: List[Tuple[np.ndarray, np.ndarray]]) -> Tuple[List, List]:
        """使用 re-materialization 进行前向传播：仅存储分区边界激活。
        
        返回：
            outputs：每个微批次的最终输出
            boundary_inputs：每个分区的输入（用于重新计算）"""
        M = len(microbatches)
        
        outputs = [None] * M
        # 只存储每个分区的输入（边界激活）
        boundary_inputs = [[None] * self.K for _ in range(M)]
        
        time_step = 0
        
        for m in range(M):
            X_micro, y_micro = microbatches[m]
            current = X_micro
            
            for k, partition in enumerate(self.partitions):
                # 将输入存储到该分区（边界）
                boundary_inputs[m][k] = current.copy()
                
                self.events.append(PipelineEvent(time_step, k, 'forward', m))
                
                # 前向传播而不存储中间激活
                current, _ = partition.forward(current, store_activations=False)
                
                time_step += 1
            
            outputs[m] = current
        
        return outputs, boundary_inputs
    
    def backward_pipeline_remat(self, outputs: List[np.ndarray],
                                labels: List[np.ndarray],
                                boundary_inputs: List[List]) -> List[List[List[Tuple]]]:
        '使用 re-materialization 反向传播：根据需要重新计算激活。'
        M = len(outputs)
        all_gradients = [[None] * self.K for _ in range(M)]
        
        time_step = max(e.time_step for e in self.events) + 1
        
        for m in range(M - 1, -1, -1):
            dout = 2 * (outputs[m] - labels[m]) / labels[m].shape[0]
            
            for k in range(self.K - 1, -1, -1):
                partition = self.partitions[k]
                
                self.events.append(PipelineEvent(time_step, k, 'backward', m))
                
                # 重新计算该分区的激活
                partition_input = boundary_inputs[m][k]
                _, activations = partition.forward(partition_input, store_activations=True)
                
                # 现在使用重新计算的激活计算梯度
                dout, gradients = partition.backward(dout, activations)
                all_gradients[m][k] = gradients
                
                time_step += 1
        
        return all_gradients


def estimate_memory_usage(M: int, K: int, layers_per_partition: int, 
                         activation_size_mb: float, with_remat: bool) -> float:
    """估计有或没有重计算的内存使用情况。
    
    参数：
        M：微批次数
        K：分区数
        layers_per_partition：每个分区的平均层数
        activation_size_mb：一层激活的内存（MB）
        with_remat：使用重计算？
    
    返回：
        估计内存（MB）"""
    if with_remat:
        # 仅存储边界输入（每个微批次 K）
        return M * K * activation_size_mb
    else:
        # 存储所有中间激活
        total_layers = K * layers_per_partition
        return M * total_layers * activation_size_mb


# 测试重计算
print("Testing re-materialization...\n")

# 使用 remat 创建新流水线
pipeline_remat = GPipePipelineWithRemat(partitions)

# 使用重计算模式执行前向传播
outputs_remat, boundary_inputs = pipeline_remat.forward_pipeline_remat(microbatches)

print("Forward pass with re-materialization:")
print(f"  Stored boundary inputs: {len(boundary_inputs)} micro-batches × {len(boundary_inputs[0])} partitions")
print(f"  Boundary input shapes: {[bi[0].shape for bi in boundary_inputs]}")

# 向后与 remat
gradients_remat = pipeline_remat.backward_pipeline_remat(outputs_remat, labels, boundary_inputs)

print(f"\nBackward pass with re-materialization:")
print(f"  Gradients computed: {len(gradients_remat)} micro-batches × {len(gradients_remat[0])} partitions")

# 内存分析
print("\n" + "="*70)
print("Memory Usage Comparison")
print("="*70)

M_test = 8
K_test = 4
layers_per_partition = 3
activation_size_mb = 10  # 每层激活 MB

mem_without = estimate_memory_usage(M_test, K_test, layers_per_partition, activation_size_mb, with_remat=False)
mem_with = estimate_memory_usage(M_test, K_test, layers_per_partition, activation_size_mb, with_remat=True)

print(f"\nConfiguration: M={M_test}, K={K_test}, {layers_per_partition} layers/partition")
print(f"  Without re-materialization: {mem_without:.1f} MB")
print(f"  With re-materialization:    {mem_with:.1f} MB")
print(f"  Memory savings:             {mem_without / mem_with:.1f}×")

print("\n✓ Re-materialization complete!")

# 第 6 节：流水线调度可视化与气泡分析

下面可视化 F-then-B 调度，并量化流水线气泡时间。

In [ ]:
def visualize_pipeline_schedule(pipeline: GPipePipeline, title: str = "GPipe Schedule (F-then-B)"):
    '可视化流水线执行时间线。'
    timeline = pipeline.get_timeline_matrix()
    K, T = timeline.shape
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # 创建颜色图
    # 正值 = 向前（暖色），负值 = 向后（冷色），0 = 气泡（白色）
    M = int(np.max(np.abs(timeline)))
    colors_forward = plt.cm.Reds(np.linspace(0.3, 0.9, M))
    colors_backward = plt.cm.Blues(np.linspace(0.3, 0.9, M))
    
    # 剧情时间线
    for k in range(K):
        for t in range(T):
            val = timeline[k, t]
            if val > 0:  # 向前
                color = colors_forward[int(val) - 1]
                label = f'F{int(val)-1}'
            elif val < 0:  # 落后
                color = colors_backward[int(-val) - 1]
                label = f'B{int(-val)-1}'
            else:  # 气泡
                color = 'white'
                label = ''
            
            rect = plt.Rectangle((t, k), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            
            if label:
                ax.text(t + 0.5, k + 0.5, label, ha='center', va='center', 
                       fontsize=9, fontweight='bold')
    
    ax.set_xlim(0, T)
    ax.set_ylim(0, K)
    ax.set_xlabel('Time Step', fontsize=12)
    ax.set_ylabel('Device', fontsize=12)
    ax.set_yticks(np.arange(K) + 0.5)
    ax.set_yticklabels([f'Device {k}' for k in range(K)])
    ax.set_xticks(np.arange(T) + 0.5)
    ax.set_xticklabels(np.arange(T))
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    # 添加图例
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='salmon', label='Forward pass'),
        Patch(facecolor='lightblue', label='Backward pass'),
        Patch(facecolor='white', edgecolor='black', label='Bubble (idle)')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()


def compute_actual_bubble_time(timeline: np.ndarray) -> float:
    '根据时间线计算实际气泡占比。'
    total_steps = timeline.size
    bubble_steps = np.sum(timeline == 0)
    return bubble_steps / total_steps


# 可视化我们之前创建的流水线
print("Visualizing GPipe pipeline schedule...\n")

visualize_pipeline_schedule(pipeline_remat, f"GPipe: K={K} devices, M={M} micro-batches")

# 分析流水线气泡时间
timeline = pipeline_remat.get_timeline_matrix()
actual_bubble = compute_actual_bubble_time(timeline)
theoretical_bubble = compute_bubble_fraction(K, M)

print(f"\nBubble Time Analysis (K={K}, M={M}):")
print(f"  Theoretical bubble fraction: {theoretical_bubble:.3f} ({theoretical_bubble*100:.1f}%)")
print(f"  Actual bubble fraction:      {actual_bubble:.3f} ({actual_bubble*100:.1f}%)")
print(f"  Pipeline efficiency:         {(1-actual_bubble)*100:.1f}%")

print("\n✓ Schedule visualization complete!")

# 第 7 节：流水线并行与数据并行对比

下面比较 GPipe 的流水线并行与传统数据并行。

## 数据并行性
- 在每台设备上复制整个模型
- 将批次拆分到多个设备
- 同步梯度（all-reduce）
- **限制**：完整模型必须能够装入单个设备

## 流水线并行（GPipe）
- 跨设备拆分模型
- 所有设备处理同一个批次中的不同微批次
- 无需梯度同步
- **优点**：可以训练单个设备内存无法容纳的模型

In [ ]:
def simulate_data_parallelism(model_layers: List[Layer], 
                             batch_size: int, 
                             num_devices: int) -> Dict[str, float]:
    """模拟数据并行时序。
    
    返回：
        带有时间细分的词典"""
    # 每个设备处理 batch_size/num_devices 示例
    local_batch_size = batch_size // num_devices
    
    # 计时（任意单位）
    forward_time = len(model_layers) * 1.0  # 每层一个单元
    backward_time = len(model_layers) * 1.0
    allreduce_time = 2.0  # 通信开销
    
    total_time = forward_time + backward_time + allreduce_time
    
    return {
        'forward': forward_time,
        'backward': backward_time,
        'communication': allreduce_time,
        'total': total_time,
        'efficiency': (forward_time + backward_time) / total_time
    }


def simulate_pipeline_parallelism(model_layers: List[Layer],
                                 batch_size: int,
                                 num_devices: int,
                                 num_microbatches: int) -> Dict[str, float]:
    '模拟流水线并行时序。'
    layers_per_device = len(model_layers) // num_devices
    
    # 一个微批次通过一个分区的时间
    forward_time_per_micro = layers_per_device * 1.0
    backward_time_per_micro = layers_per_device * 1.0
    
    # 总流水线时间
    # 填充流水线：(K-1)+M微批次
    # 每一步：向前或向后穿过一个分区
    total_forward_steps = (num_devices - 1) + num_microbatches
    total_backward_steps = (num_devices - 1) + num_microbatches
    
    total_time = (total_forward_steps + total_backward_steps) * layers_per_device
    
    # 计算时间（不包括气泡）
    compute_time = 2 * num_microbatches * layers_per_device * num_devices
    
    return {
        'forward': total_forward_steps * layers_per_device,
        'backward': total_backward_steps * layers_per_device,
        'communication': 0,  # 没有设备间通信！
        'total': total_time,
        'efficiency': compute_time / (total_time * num_devices),
        'bubble_fraction': compute_bubble_fraction(num_devices, num_microbatches)
    }


# 比较两种方法
print("Comparing Pipeline Parallelism vs Data Parallelism\n")
print("="*70)

total_layers = 12
batch_size = 32
num_devices = 4
num_microbatches = 8

# 模拟数据并行性
data_parallel_stats = simulate_data_parallelism(model_layers, batch_size, num_devices)

print("Data Parallelism:")
print(f"  Configuration: {num_devices} devices, batch size {batch_size}")
print(f"  Forward time:        {data_parallel_stats['forward']:.1f} units")
print(f"  Backward time:       {data_parallel_stats['backward']:.1f} units")
print(f"  Communication time:  {data_parallel_stats['communication']:.1f} units (all-reduce)")
print(f"  Total time:          {data_parallel_stats['total']:.1f} units")
print(f"  Efficiency:          {data_parallel_stats['efficiency']*100:.1f}%")
print(f"  ⚠️  Limitation: Model must fit on single device!")

print("\n" + "="*70)

# 模拟流水线并行性
pipeline_stats = simulate_pipeline_parallelism(model_layers, batch_size, num_devices, num_microbatches)

print("Pipeline Parallelism (GPipe):")
print(f"  Configuration: {num_devices} devices, {num_microbatches} micro-batches")
print(f"  Forward time:        {pipeline_stats['forward']:.1f} units")
print(f"  Backward time:       {pipeline_stats['backward']:.1f} units")
print(f"  Communication time:  {pipeline_stats['communication']:.1f} units (none!)")
print(f"  Total time:          {pipeline_stats['total']:.1f} units")
print(f"  Efficiency:          {pipeline_stats['efficiency']*100:.1f}%")
print(f"  Bubble fraction:     {pipeline_stats['bubble_fraction']*100:.1f}%")
print(f"  ✓ Advantage: Can train models {num_devices}× larger!")

print("\n" + "="*70)
print("\nKey Differences:")
print("  • Data parallel: Fast, but model must fit on one device")
print("  • Pipeline parallel: Enables training of giant models")
print("  • GPipe: No communication overhead (unlike data parallel)")
print("  • Trade-off: Pipeline has bubble time, data parallel has communication")

print("\n✓ Comparison complete!")

# 第 8 节：完整的 GPipe 训练循环

下面将前面的组件组合成一个完整的 GPipe 训练循环。

In [ ]:
def compute_loss(outputs: List[np.ndarray], labels: List[np.ndarray]) -> float:
    '计算微批次的平均损失（为了简单起见，MSE）。'
    total_loss = 0.0
    for output, label in zip(outputs, labels):
        total_loss += np.mean((output - label) ** 2)
    return total_loss / len(outputs)


def train_gpipe_epoch(pipeline: GPipePipelineWithRemat,
                     X_train: np.ndarray,
                     y_train: np.ndarray,
                     batch_size: int,
                     num_microbatches: int,
                     learning_rate: float) -> List[float]:
    """使用 GPipe 训练一个 epoch。
    
    返回：
        每个小批量的损失列表"""
    num_samples = X_train.shape[0]
    num_batches = num_samples // batch_size
    
    losses = []
    
    for batch_idx in range(num_batches):
        # 获取小批量
        start = batch_idx * batch_size
        end = start + batch_size
        X_batch = X_train[start:end]
        y_batch = y_train[start:end]
        
        # 分成微批次
        microbatches = split_into_microbatches(X_batch, y_batch, num_microbatches)
        
        # 前向传播
        outputs, boundary_inputs = pipeline.forward_pipeline_remat(microbatches)
        
        # 计算损失
        labels = [mb[1] for mb in microbatches]
        loss = compute_loss(outputs, labels)
        losses.append(loss)
        
        # 向后传球
        all_gradients = pipeline.backward_pipeline_remat(outputs, labels, boundary_inputs)
        
        # 累积梯度
        accumulated_grads = accumulate_gradients(all_gradients)
        
        # 更新参数
        apply_gradients(pipeline.partitions, accumulated_grads, learning_rate)
    
    return losses


# 生成综合数据集
print("Creating synthetic dataset...\n")

num_train = 256
input_dim = 128
output_dim = 10

X_train = np.random.randn(num_train, input_dim)
y_train_labels = np.random.randint(0, output_dim, num_train)
y_train = np.eye(output_dim)[y_train_labels]

print(f"Dataset: {num_train} samples, input dim {input_dim}, output dim {output_dim}")

# 创建新的模型和流水线
print("\nInitializing GPipe model...")

layer_dims = [input_dim] + [256] * 10 + [output_dim]
activations = ['relu'] * 10 + ['linear']
model_layers = create_model(layer_dims, activations)

K = 4
partitions = partition_model(model_layers, K)
pipeline = GPipePipelineWithRemat(partitions)

print(f"  Model: {len(model_layers)} layers")
print(f"  Partitions: {K} devices")

# 训练配置
batch_size = 32
num_microbatches = 8
learning_rate = 0.001
num_epochs = 3

print(f"\nTraining configuration:")
print(f"  Batch size: {batch_size}")
print(f"  Micro-batches: {num_microbatches}")
print(f"  Learning rate: {learning_rate}")
print(f"  Epochs: {num_epochs}")

# 火车
print("\n" + "="*70)
print("Training GPipe model...")
print("="*70 + "\n")

all_losses = []

for epoch in range(num_epochs):
    pipeline.events = []  # 重置该纪元的事件
    
    losses = train_gpipe_epoch(pipeline, X_train, y_train, 
                               batch_size, num_microbatches, learning_rate)
    
    avg_loss = np.mean(losses)
    all_losses.extend(losses)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Average Loss = {avg_loss:.6f}")

print("\n✓ Training complete!")

# 第 9 节：可视化和分析

下面从多个角度可视化并分析 GPipe 的性能。

In [ ]:
# 可视化 1：训练损失曲线
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图 1：训练损失
ax = axes[0, 0]
ax.plot(all_losses, linewidth=2, color='darkblue')
ax.set_xlabel('Mini-batch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('GPipe Training Loss', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 图 2：气泡占比与 M（微批次）
ax = axes[0, 1]
M_range = np.arange(1, 65)
K_values_plot = [2, 4, 8, 16]
colors = ['blue', 'green', 'orange', 'red']

for K_val, color in zip(K_values_plot, colors):
    bubbles = [compute_bubble_fraction(K_val, M) for M in M_range]
    ax.plot(M_range, bubbles, label=f'K={K_val}', linewidth=2, color=color)

ax.set_xlabel('Number of Micro-batches (M)', fontsize=11)
ax.set_ylabel('Bubble Fraction', fontsize=11)
ax.set_title('Bubble Time vs Micro-batches', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

# 图 3：通过重计算节省内存
ax = axes[1, 0]
K_range = np.arange(2, 17)
layers_per_partition = 3
M_fixed = 8
activation_size_mb = 10

mem_without_remat = [estimate_memory_usage(M_fixed, K_val, layers_per_partition, 
                                            activation_size_mb, False) 
                     for K_val in K_range]
mem_with_remat = [estimate_memory_usage(M_fixed, K_val, layers_per_partition, 
                                        activation_size_mb, True) 
                  for K_val in K_range]

ax.plot(K_range, mem_without_remat, label='Without Remat', linewidth=2, 
        marker='o', color='red', markersize=6)
ax.plot(K_range, mem_with_remat, label='With Remat', linewidth=2, 
        marker='s', color='green', markersize=6)
ax.set_xlabel('Number of Partitions (K)', fontsize=11)
ax.set_ylabel('Memory (MB)', fontsize=11)
ax.set_title('Memory Usage: Re-materialization Impact', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 图 4：流水线效率与配置
ax = axes[1, 1]
M_configs = [4, 8, 16, 32]
K_configs = np.arange(2, 17)

for M_val in M_configs:
    efficiencies = [1 - compute_bubble_fraction(K_val, M_val) for K_val in K_configs]
    ax.plot(K_configs, efficiencies, label=f'M={M_val}', linewidth=2, marker='o', markersize=5)

ax.set_xlabel('Number of Devices (K)', fontsize=11)
ax.set_ylabel('Pipeline Efficiency', fontsize=11)
ax.set_title('Pipeline Efficiency vs Configuration', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\n✓ Visualizations complete!")

# 第 10 节：核心认识与现代扩展

## GPipe 总结

### 核心理念
1. **流水线并行**：按层将模型拆分到多个设备
2. **微批处理**：拆分小批次以减少气泡时间
3. **重计算**：用额外计算换取更高的内存效率
4. **F-then-B 调度**：先完成所有微批次的前向传播，再完成全部反向传播

### 数学见解

**气泡占比**：
$$\text{Bubble} = \frac{K-1}{K-1+M}$$

**内存节省**（重计算）：
$$\text{Memory}_{\text{remat}} = \frac{K}{L} \times \text{Memory}_{\text{standard}}$$

其中 L 表示总层数，K 表示分区数。

**加速**（与单个设备相比）：
$$\text{Speedup} \approx \frac{K}{1 + \frac{K-1}{M}}$$

### 何时使用 GPipe

**适合使用 GPipe 的情况：**
- 模型无法装入单个设备
- 顺序模型结构（层）
- 设备间带宽有限
- 可以使用较大的 M，也就是较多微批次

**不适合使用 GPipe 的情况：**
- 模型能够装入单个设备，此时可优先考虑数据并行
- M 很小，流水线气泡占据主要时间
- 网络不是顺序结构，例如包含大量跳跃连接

---

## 现代扩展

### 1. PipeDream（Harlap 等人，2018）
- **1F1B 调度**：交错执行前向传播与反向传播
- 减少流水线气泡
- 更好的内存效率

### 2. Megatron-LM（Shoeybi et al., 2019）
- 结合流水线并行与张量并行
- 在单层内部沿水平方向拆分计算
- 用于训练 530B 参数规模的模型

### 3. ZeRO（Rajbhandari et al., 2020）
- 分区优化器状态、梯度、参数
- 可以与流水线并行互补
- 通过消除不必要的复制降低内存占用

### 4. Varuna（Athlur et al., 2022）
- 自动流水线调度优化
- 自适应微批处理
- 处理异构设备

---

## 实践注意事项

### 最优 M（微批次）
- **太小**：气泡占比高
- **太大**：微批次管理开销过高
- **经验法则**：M ≈ 4×K

### 分区策略
- 均匀分区：每台设备的层数相同
- 平衡：每个设备的计算时间相等
- 内存感知：平衡内存使用

### 批次大小
- 大批次可以提高流水线利用率
- 但可能会损害泛化
- 通过学习率缩放进行补偿

---

## 与其他论文的联系

**论文 5（Optimal Brain Damage）**：剪枝减小模型规模 → 所需流水线阶段更少

**论文 23 (MDL)**：模型复杂性与数据拟合 → 选择 K（分区）涉及权衡

**论文 14（神经架构搜索）**：可以使用 GPipe 搜索对于单个设备来说太大的架构

---

## 现实世界的影响

GPipe 带来的实际成果包括：
- **AmoebaNet-B**：5.57 亿参数（比此前最佳模型大 8 倍）
- **在 ImageNet 上训练**，top-1 准确度为 84.4%
- **GPT-3**：175B 参数，训练中组合使用了包括流水线并行在内的多种技术
- **大型语言模型**：现代 LLM 通常结合流水线并行、张量并行和数据并行

---

**GPipe 的影响**：它证明了**模型并行具有实用价值**，并为训练数千亿参数规模的模型铺平道路。流水线并行与张量并行、ZeRO 等技术相结合，构成了现代大规模训练的重要基础。

In [ ]:
# 最终演示：展示 K 和 M 之间的权衡
print("="*70)
print("GPipe Configuration Guide")
print("="*70)

print("\n1. Choosing K (number of devices):")
print("   • Limited by: Number of available accelerators")
print("   • More K = Can train larger models")
print("   • More K = More bubble time (need larger M to compensate)")

print("\n2. Choosing M (number of micro-batches):")
print("   • Rule of thumb: M ≈ 4×K")
print("   • Larger M = Less bubble time")
print("   • Larger M = More overhead")
print("   • Must divide batch size evenly")

print("\n3. Example configurations:")
configs = [
    (2, 8, 32),
    (4, 16, 64),
    (8, 32, 128),
    (16, 64, 256),
]

for K, M, batch in configs:
    bubble = compute_bubble_fraction(K, M)
    efficiency = 1 - bubble
    print(f"   K={K:2d}, M={M:2d}, batch={batch:3d} → "
          f"Efficiency={efficiency*100:.1f}%, Bubble={bubble*100:.1f}%")

print("\n" + "="*70)
print("✓ GPipe implementation complete!")
print("="*70)